# 근거 승인과 보고서 작성

규칙 Supervisor가 MCP 조사·승인·작성·평가·종료를 선택합니다. 기본 Writer는 원문 발췌이며 실제 모델을 넘겨줄 때만 LLM을 호출합니다. 승인 재개는 같은 thread, 새 요청은 새 thread를 사용합니다.

구현 정본은 [capstone.py](capstone.py)입니다. 아래에서 현재 파일을 읽고 import하므로 Python 수정이 자동으로 반영됩니다. 수업 본문은 [통합 교안](../../lessons/)을 사용하세요.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/courseware").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"


In [ ]:
from IPython.display import display, Code
display(Code((ROOT / "src/courseware/capstone.py").read_text(encoding="utf-8"), language="python"))

## 작은 실행과 검토

입력·출력·실패 조건을 예상하고 실행하세요. 원문 코드의 구현과 실제 결과를 대조합니다.

In [ ]:
from courseware.capstone import build_capstone
from langgraph.types import Command
from uuid import uuid4
for approved in [True, False]:
    graph = build_capstone()
    config = {"configurable": {"thread_id": str(uuid4())}}
    pending = await graph.ainvoke({"topic": "RAG 검색", "trace": []}, config)
    assert pending.get("__interrupt__")
    result = await graph.ainvoke(Command(resume={"approved": approved}), config)
    assert result["writer_executed"] is approved
    print("승인", approved, "→", result["trace"])
    if approved:
        print(result["report"])